Public - 0.56748	Private - 0.62783

In [1]:
# ==========================================
# CELL 1: SETUP & EXACT PATH RESOLUTION
# ==========================================
import os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision

try:
    from ultralytics import YOLO
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)
    from ultralytics import YOLO

# 1. Hardcoded paths based on your specific Kaggle environment
INPUT_ROOT = Path("/kaggle/input/competitions/road-vision/RoadVision_DUET")

# We use rglob for the CSV in case it is sitting in the root or a subfolder
TRAIN_CSV = next(iter(INPUT_ROOT.rglob("train.csv")), None)

# Your exact provided paths
SAMPLE_SUB = INPUT_ROOT / "test" / "sample_submission.csv"
TRAIN_IMAGES = INPUT_ROOT / "train" / "images"
TEST_IMAGES  = INPUT_ROOT / "test" / "images"

print(f"Train CSV path resolved: {TRAIN_CSV}")
print(f"Train Images path resolved: {TRAIN_IMAGES}")
print(f"Test Images path resolved: {TEST_IMAGES}")

assert TRAIN_CSV and TRAIN_IMAGES.exists() and TEST_IMAGES.exists(), "Path mapping failed! Check directory structure."

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 34.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Train CSV path resolved: /kaggle/input/competitions/road-vision/RoadVision_DUET/train/train.csv
Train Images path resolved: /kaggle/input/competitions/road-vision/RoadVision_DUET/train/images
Test Images path resolved: /kaggle/input/competitions/road-vision/RoadVision_DUET/test/images


In [2]:
# ==========================================
# CELL 2: K-FOLD CONFIGURATION
# ==========================================
K_FOLDS = 5
EPOCHS_PER_FOLD = 40  # Reduced slightly from 60 to ensure all 5 folds finish within Kaggle's 9-hour limit
MODEL_ARCH = "yolov8s.pt" # Or yolo11s.pt depending on your specific Ultralytics version
IMGSZ = 960
BATCH = 16
DEVICE = 0

# Inference settings
PRED_CONF = 0.001 
PRED_IOU = 0.60
ENSEMBLE_IOU = 0.65 # The IoU threshold for the final voting mechanism

BASE_TEMP_DIR = Path("/kaggle/temp/roadvision_kfold")
RUNS_DIR = Path("/kaggle/working/runs")

In [3]:
# ==========================================
# CELL 3: TEMPORAL CHUNK SPLITTING & COMPILATION
# ==========================================
df = pd.read_csv(TRAIN_CSV)
df['stem'] = df['image_id'].str.replace('.txt', '', regex=False)

# Index images
train_idx = {p.stem: p for p in TRAIN_IMAGES.rglob("*.*") if p.suffix.lower() in ['.jpg', '.png', '.jpeg']}
df = df[df['stem'].isin(train_idx)].reset_index(drop=True)
stems = sorted(df['stem'].unique())

# Group by Camera Clip
by_clip = {}
for s in stems:
    clip_id = s.split('_')[0]
    by_clip.setdefault(clip_id, []).append(s)

# Create K Folds via continuous chunking
folds_val_stems = {i: [] for i in range(K_FOLDS)}

for clip, clip_stems in by_clip.items():
    clip_stems.sort() # Ensure chronological order
    # Split this clip into K roughly equal continuous chunks
    chunks = np.array_split(clip_stems, K_FOLDS)
    for i in range(K_FOLDS):
        folds_val_stems[i].extend(chunks[i])

print(f"Mathematical Split Complete. Total unique images: {len(stems)}")
for i in range(K_FOLDS):
    print(f"Fold {i} Validation Images: {len(folds_val_stems[i])}")

# Group annotations
groups = {k: g for k, g in df.groupby("stem")}

def compile_fold(fold_idx):
    """Compiles the YOLO directory matrix for a specific fold."""
    fold_dir = BASE_TEMP_DIR / f"fold_{fold_idx}"
    if fold_dir.exists(): shutil.rmtree(fold_dir)
    
    for split in ["train", "val"]:
        (fold_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (fold_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        
    val_set = set(folds_val_stems[fold_idx])
    
    for stem in stems:
        split = "val" if stem in val_set else "train"
        src = train_idx[stem]
        dst_img = fold_dir / "images" / split / src.name
        
        # O(1) Symlinking
        os.symlink(src.resolve(), dst_img)
        
        # Write annotation tensor
        g = groups[stem]
        lines = [f"{int(r['class_id'])} {r['x_center']:.6f} {r['y_center']:.6f} {r['width']:.6f} {r['height']:.6f}"
                 for _, r in g.iterrows()]
        (fold_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
        
    # Generate YAML
    yaml_text = f"path: {fold_dir}\ntrain: images/train\nval: images/val\nnc: 13\nnames: {[str(i) for i in range(13)]}\n"
    (fold_dir / "data.yaml").write_text(yaml_text)
    return str(fold_dir / "data.yaml")

# Compile all folds to disk instantly
fold_yamls = [compile_fold(i) for i in range(K_FOLDS)]
print("All K-Fold directories compiled via symlinks.")

Mathematical Split Complete. Total unique images: 810
Fold 0 Validation Images: 164
Fold 1 Validation Images: 163
Fold 2 Validation Images: 163
Fold 3 Validation Images: 160
Fold 4 Validation Images: 160
All K-Fold directories compiled via symlinks.


In [4]:
# ==========================================
# CELL 4: SEQUENTIAL TRAINING LOOP
# ==========================================
print("Initiating K-Fold Training Matrix...")

for fold in range(K_FOLDS):
    print(f"\n{'='*40}\nCOMMENCING FOLD {fold}\n{'='*40}")
    model = YOLO(MODEL_ARCH)
    
    model.train(
        data=fold_yamls[fold],
        epochs=EPOCHS_PER_FOLD,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=RUNS_DIR,
        name=f"fold_{fold}",
        mosaic=1.0,
        close_mosaic=10,
        verbose=False # Keep logs clean
    )
    print(f"Fold {fold} complete. Weights mapped to: {RUNS_DIR}/fold_{fold}/weights/best.pt")

Initiating K-Fold Training Matrix...

COMMENCING FOLD 0
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/temp/roadvision_kfold/fold_0/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=40, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_0, nbs=64, nms=False, opset

In [5]:
# ==========================================
# CELL 5: ENSEMBLE INFERENCE & SUBMISSION
# ==========================================
test_idx = {p.stem: p for p in TEST_IMAGES.rglob("*.*") if p.suffix.lower() in ['.jpg', '.png', '.jpeg']}
test_paths = sorted(list(test_idx.values()))

print("\nLoading the K-Fold Ensemble...")
# Load all 5 optimized models into VRAM simultaneously
models = [YOLO(f"{RUNS_DIR}/fold_{i}/weights/best.pt") for i in range(K_FOLDS)]

sample_df = pd.read_csv(SAMPLE_SUB)
sample_df['image_id'] = sample_df['image_id'].str.replace('.txt', '', regex=False)
predictions_dict = {}

print(f"Executing Consensus Forward Pass on {len(test_paths)} images...")

for img_path in test_paths:
    stem = img_path.stem
    all_boxes, all_scores, all_classes = [], [], []
    
    # 1. Forward Pass through all 5 models
    for model in models:
        # We use a slightly higher confidence here before ensembling to save memory
        results = model.predict(source=str(img_path), imgsz=IMGSZ, conf=0.01, verbose=False)[0]
        b = results.boxes
        if b is not None and len(b) > 0:
            # Extract absolute coordinates [x1, y1, x2, y2] for PyTorch NMS
            all_boxes.append(b.xyxy.cpu()) 
            all_scores.append(b.conf.cpu())
            all_classes.append(b.cls.cpu())
            
    if len(all_boxes) == 0:
        predictions_dict[stem] = ""
        continue
        
    # Concatenate all fold tensors into one massive swarm
    all_boxes = torch.cat(all_boxes, dim=0)
    all_scores = torch.cat(all_scores, dim=0)
    all_classes = torch.cat(all_classes, dim=0)
    
    pred_string_parts = []
    
    # 2. Per-Class Consensus Voting (NMS)
    for c in torch.unique(all_classes):
        mask = (all_classes == c)
        c_boxes = all_boxes[mask]
        c_scores = all_scores[mask]
        
        # Mathematically vote: NMS keeps the highest confidence box and suppresses overlaps
        keep_idx = torchvision.ops.nms(c_boxes, c_scores, ENSEMBLE_IOU)
        
        final_boxes = c_boxes[keep_idx]
        final_scores = c_scores[keep_idx]
        
        # Format the surviving consensus boxes back into YOLO [x_c, y_c, w, h] normalized
        # Image dimensions are needed to normalize back to [0, 1]
        img_w, img_h = results.orig_shape[1], results.orig_shape[0]
        
        for box, score in zip(final_boxes, final_scores):
            x1, y1, x2, y2 = box.tolist()
            # Convert to [0, 100] confidence metric
            conf_percent = float(score) * 100.0 
            
            # Translate absolute corners to normalized centers
            xc = ((x1 + x2) / 2.0) / img_w
            yc = ((y1 + y2) / 2.0) / img_h
            w  = (x2 - x1) / img_w
            h  = (y2 - y1) / img_h
            
            pred_string_parts.extend([
                str(int(c.item())), 
                f"{conf_percent:.2f}", 
                f"{xc:.6f}", f"{yc:.6f}", f"{w:.6f}", f"{h:.6f}"
            ])
            
    predictions_dict[stem] = " ".join(pred_string_parts)

# 3. Surjective Projection to match exact Kaggle row ordering
sample_df['PredictionString'] = sample_df['image_id'].apply(lambda x: predictions_dict.get(x, ""))
output_csv = "/kaggle/working/submission.csv"
sample_df.to_csv(output_csv, index=False)
print(f"\nEnsemble Complete! Saved to {output_csv}")


Loading the K-Fold Ensemble...
Executing Consensus Forward Pass on 327 images...

Ensemble Complete! Saved to /kaggle/working/submission.csv
